# Output Token Length Analysis

In [ ]:
import numpy as np
import pandas as pd
import datasets
from transformers import AutoTokenizer

MODEL_PATHS = {
    "AceGPT-7B": "/raid_storage/shared_models/AceGPT-7B",
    "Meta-Llama-3.1-8B": "/raid_storage/shared_models/Meta-Llama-3.1-8B",
    "Qwen3-8B": "/raid_storage/shared_models/Qwen3-8B-Base",
}

DATASETS_CONFIG = {
    "summarization": {
        "xlsum": {
            "hf_path": "KFUPM-JRCAI/xlsum_arabic_experimental",
            "output_col": "summary",
            "splits": ["train", "test"],
        },
        "AraSum": {
            "hf_path": "KFUPM-JRCAI/AraSum_arabic_experimental",
            "output_col": "summary",
            "splits": ["train", "test"],
        },
    },
    "machine_translation": {
        "opus-100": {
            "hf_path": "KFUPM-JRCAI/opus-100_ar_en_experimental",
            "output_col": "ar",
            "splits": ["train", "test"],
        },
        "tatoeba_mt": {
            "hf_path": "KFUPM-JRCAI/tatoeba_mt_ara_eng_experimental",
            "output_col": "ar",
            "splits": ["train", "test"],
        },
    },
}

## Load Tokenizers

In [ ]:
tokenizers = {}
for name, path in MODEL_PATHS.items():
    print(f"Loading tokenizer: {name}")
    tokenizers[name] = AutoTokenizer.from_pretrained(path)
print("All tokenizers loaded.")

## Load Datasets and Compute Token Lengths

In [ ]:
results = []

for task, ds_configs in DATASETS_CONFIG.items():
    for ds_name, ds_info in ds_configs.items():
        print(f"Loading {task}/{ds_name}...")
        hf_ds = datasets.load_dataset(ds_info["hf_path"])
        output_col = ds_info["output_col"]

        for split in ds_info["splits"]:
            if split not in hf_ds:
                print(f"  Skipping split '{split}' (not found)")
                continue

            texts = hf_ds[split][output_col]
            print(f"  {split}: {len(texts)} samples")

            for model_name, tokenizer in tokenizers.items():
                token_lengths = [
                    len(tokenizer.encode(text)) for text in texts
                ]
                results.append({
                    "task": task,
                    "dataset": ds_name,
                    "split": split,
                    "model": model_name,
                    "token_lengths": token_lengths,
                })

print("Done.")

## Statistics: Mean, Median, P90, P95, P99

In [ ]:
stats_rows = []
for r in results:
    lengths = np.array(r["token_lengths"])
    stats_rows.append({
        "Task": r["task"],
        "Dataset": r["dataset"],
        "Split": r["split"],
        "Tokenizer": r["model"],
        "Count": len(lengths),
        "Mean": round(np.mean(lengths), 1),
        "Median": round(np.median(lengths), 1),
        "P90": round(np.percentile(lengths, 90), 1),
        "P95": round(np.percentile(lengths, 95), 1),
        "P99": round(np.percentile(lengths, 99), 1),
        "Max": int(np.max(lengths)),
    })

stats_df = pd.DataFrame(stats_rows)
stats_df

## Summarization Stats Only

In [ ]:
stats_df[stats_df["Task"] == "summarization"]

## Machine Translation Stats Only

In [ ]:
stats_df[stats_df["Task"] == "machine_translation"]

## Aggregated by Task (across all datasets, splits, and tokenizers)

In [ ]:
for task in ["summarization", "machine_translation"]:
    task_results = [r for r in results if r["task"] == task]
    all_lengths = np.concatenate([r["token_lengths"] for r in task_results])
    print(f"=== {task.upper()} (all datasets, splits, tokenizers combined) ===")
    print(f"  Total samples (tokenized): {len(all_lengths)}")
    print(f"  Mean:   {np.mean(all_lengths):.1f}")
    print(f"  Median: {np.median(all_lengths):.1f}")
    print(f"  P90:    {np.percentile(all_lengths, 90):.1f}")
    print(f"  P95:    {np.percentile(all_lengths, 95):.1f}")
    print(f"  P99:    {np.percentile(all_lengths, 99):.1f}")
    print(f"  Max:    {int(np.max(all_lengths))}")
    print()